In [ ]:
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True


Device: cuda


In [ ]:
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler() if use_amp else None

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc="train", leave=False)

    for i, (x, y) in enumerate(pbar):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        if use_amp:
            with torch.cuda.amp.autocast():
                output = model(x)
                loss = loss_fn(output, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(x)
            loss = loss_fn(output, y)

            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=total_loss / (i + 1))

    return total_loss / len(loader)


/tmp/ipykernel_6792/3962349144.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


In [ ]:
import os
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, SubsetRandomSampler, random_split
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from torchvision.datasets import EMNIST
from torchvision import transforms

In [ ]:
os.makedirs('artifacts/figures', exist_ok=True)

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_set = EMNIST('data/', split='balanced', download=True,
transform=transforms.Compose([
transforms.ToTensor(),
]), train=True)


test_set = EMNIST('data/', split='balanced', download=False,
transform=transforms.Compose([
transforms.ToTensor(),
]), train=False)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = train_set
test_dataset = test_set

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


100%|██████████| 562M/562M [00:04<00:00, 125MB/s]


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, use_dropout=False, dropout_p=0.2, use_bn=False):
        super(MLP, self).__init__()
        layers = []
        prev_size = input_size

        for i, hidden_size in enumerate(hidden_sizes): # test count
            layers.append(nn.Linear(prev_size, hidden_size))
            if use_bn:
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(p=dropout_p))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

In [ ]:
def run_experiment(exp_id, model_config, opt_config, epochs, early_stopping_patience=0):
    model = MLP(
        input_size=28*28,
        hidden_sizes=model_config['hidden_sizes'],
        output_size=47,
        use_dropout=model_config.get('use_dropout', False),
        dropout_p=model_config.get('dropout_p', 0.0),
        use_bn=model_config.get('use_bn', False)
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if opt_config['type'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=opt_config['lr'], weight_decay=opt_config.get('weight_decay', 0))
    elif opt_config['type'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=opt_config['lr'], momentum=opt_config.get('momentum', 0), weight_decay=opt_config.get('weight_decay', 0))

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    actual_epochs = epochs

    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping_patience > 0 and patience_counter >= early_stopping_patience:
            actual_epochs = epoch + 1
            break

    return {
        'experiment_id': exp_id,
        'history': history,
        'best_val_accuracy': best_val_acc,
        'best_val_loss': min(history['val_loss']),
        'epochs_trained': actual_epochs,
        'model_state': best_model_state,
        'model_config': model_config,
        'opt_config': opt_config
    }

In [ ]:
# EMNIST balanced has 47 classes (output_size=47 set in run_experiment)

results = []

hidden_sizes = [512, 256, 128]

model_base = {'hidden_sizes': hidden_sizes, 'use_dropout': False, 'use_bn': False}
model_dropout = {'hidden_sizes': hidden_sizes, 'use_dropout': True, 'dropout_p': 0.3, 'use_bn': False}
model_bn = {'hidden_sizes': hidden_sizes, 'use_dropout': False, 'use_bn': True}

opt_adam = {'type': 'Adam', 'lr': 1e-3, 'weight_decay': 0}
opt_sgd = {'type': 'SGD', 'lr': 1e-2, 'momentum': 0.9, 'weight_decay': 1e-4}

epochs_default = 20

res_e1 = run_experiment('E1', model_base, opt_adam, epochs_default)
results.append(res_e1)

res_e2 = run_experiment('E2', model_dropout, opt_adam, epochs_default)
results.append(res_e2)

res_e3 = run_experiment('E3', model_bn, opt_adam, epochs_default)
results.append(res_e3)

best_e2e3 = res_e2 if res_e2['best_val_accuracy'] > res_e3['best_val_accuracy'] else res_e3
best_model_config = best_e2e3['model_config']

res_e4 = run_experiment('E4', best_model_config, opt_adam, epochs_default, early_stopping_patience=5)
results.append(res_e4)

In [ ]:
opt_high_lr = {'type': 'Adam', 'lr': 1e-1, 'weight_decay': 0}
opt_low_lr = {'type': 'Adam', 'lr': 1e-5, 'weight_decay': 0}

res_o1 = run_experiment('O1', model_base, opt_high_lr, 8)
results.append(res_o1)
res_o2 = run_experiment('O2', model_base, opt_low_lr, 8)
results.append(res_o2)
res_o3 = run_experiment('O3', model_base, opt_sgd, 15)
results.append(res_o3)


In [ ]:
runs_data = []
for r in results:
    runs_data.append({
        'experiment_id': r['experiment_id'],
        'dataset': 'EMNIST',
        'seed': seed,
        'model_summary': f"{r['model_config']['hidden_sizes']}/ReLU/DP{r['model_config'].get('use_dropout', False)}/BN{r['model_config'].get('use_bn', False)}",
        'optimizer': r['opt_config']['type'],
        'lr': r['opt_config']['lr'],
        'momentum': r['opt_config'].get('momentum', 0),
        'weight_decay': r['opt_config'].get('weight_decay', 0),
        'epochs_trained': r['epochs_trained'],
        'best_val_accuracy': r['best_val_accuracy'],
        'best_val_loss': r['best_val_loss']
    })

df_runs = pd.DataFrame(runs_data)
df_runs.to_csv('artifacts/runs.csv', index=False)


In [ ]:
best_result = res_e4
torch.save(best_result['model_state'], 'artifacts/best_model.pt')

config_save = {
    'dataset': 'EMNIST',
    'seed': seed,
    'model_config': best_result['model_config'],
    'opt_config': best_result['opt_config'],
    'epochs_trained': best_result['epochs_trained']
}
with open('artifacts/best_config.json', 'w') as f:
    json.dump(config_save, f, indent=2)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(best_result['history']['train_loss'], label='Train Loss')
plt.plot(best_result['history']['val_loss'], label='Val Loss')
plt.title(f'E4 Best Model Loss (Val Acc: {best_result["best_val_accuracy"]:.4f})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('artifacts/figures/curves_best.png')
plt.close()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(res_o1['history']['val_loss'], label='O1 High LR (1e-1)', linestyle='--')
plt.plot(res_o2['history']['val_loss'], label='O2 Low LR (1e-5)', linestyle='-.')
plt.title('LR Extremes Diagnosis')
plt.xlabel('Epoch')
plt.ylabel('Val Loss')
plt.legend()
plt.grid(True)
plt.savefig('artifacts/figures/curves_lr_extremes.png')
plt.close()


In [ ]:
final_model = MLP(
    input_size=28*28,
    hidden_sizes=best_result['model_config']['hidden_sizes'],
    output_size=47,
    use_dropout=best_result['model_config'].get('use_dropout', False),
    dropout_p=best_result['model_config'].get('dropout_p', 0.0),
    use_bn=best_result['model_config'].get('use_bn', False)
).to(device)
final_model.load_state_dict(best_result['model_state'])

test_loss, test_acc = evaluate(final_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"Final Test Accuracy: {test_acc:.4f}")

Final Test Accuracy: 0.8504
